In [1]:
#!/g/data/xp65/public/apps/med_conda_scripts/analysis3-25.07.d/bin/python3
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import datetime
import xesmf as xe

from dask.distributed import Client, wait
import os, sys

sys.path.append('/home/548/cd3022/repos/Irradiance-comparisons/Irradiance-comparisons')
import logger
from read_datasets import read_dataset

date = '2020-01'
LOG = logger.get_logger(__name__)
# qsub -I -q normal -P er8 -l walltime=2:00:00,ncpus=24,mem=120GB,jobfs=100MB,storage=gdata/xp65+gdata/er8+gdata/ob53+gdata/rt52+gdata/rv74+gdata/su28

def get_diff(ds1, ds2):
    return ds2.ghi - ds1.ghi

def calc_rmse(diff):
    LOG.info('start calc rmse')
    rmse = np.sqrt((np.square(diff)).weighted(np.cos(np.deg2rad(diff.lat))).mean(['lat', 'lon']))
    LOG.info('rmse calculated')
    rmse.name = 'rmse'
    return rmse

def rmse_workflow(ds1, ds2):
    diff = get_diff(ds1, ds2)
    LOG.info('diff calculated')
    return calc_rmse(diff)

In [2]:
client = Client(
    n_workers=6,
    threads_per_worker=4,
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 6
Total threads: 24,Total memory: 46.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40807,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38325,Total threads: 4
Dashboard: /proxy/39047/status,Memory: 7.67 GiB
Nanny: tcp://127.0.0.1:45445,


In [10]:
%%time
ds1_list = []
ds2_list = []
year=2023
for month in range(1,13):
    date = f'{year}-{month:02d}'
    ds1 = read_dataset(
            dataset='himawari',
            resolution='daily',
            date=date
        )
    ds1_list.append(ds1)
    ds2 = read_dataset(
            dataset='barra-r2',
            resolution='daily',
            date=date
        )
    ds2_list.append(ds2)
ds1 = xr.concat(ds1_list, dim='time')
ds2 = xr.concat(ds2_list, dim='time')

ds1 = ds1.assign_coords(time=ds1.time + np.timedelta64(1, "D"))
ds1 = ds1.assign_coords(time=ds1.time.dt.floor("D"))
ds2 = ds2.assign_coords(time=ds2.time.dt.floor("D"))

regridder_file = '/g/data/er8/users/cd3022/regridder_weights/himawari_to_barrar2_weights.nc'
regridder =  xe.Regridder(ds1, ds2, "bilinear",
                     filename=regridder_file,
                     reuse_weights=True)
ds1 = regridder(ds1)
rmse = rmse_workflow(ds1, ds2)

file_path = Path(f'/g/data/er8/users/cd3022/Irradiance-comparisons/error_timeseries/')
os.makedirs(file_path, exist_ok=True)
rmse.to_netcdf(f'{file_path}/himawari-barrar2_{year}.nc')

2025-09-09 14:54:11,304:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:447: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 16.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (20, 2214).
  result_vars[name] = func(*variable_args)

2025-09-09 14:54:11,304:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:447: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 16.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (20, 2214).
  result_vars[name] = func(*variable_args)

  result_vars[name] = func(*variable_args)

2025-09-09 14:54:11,309:py.warnings:WARNING: /g/data/xp6

CPU times: user 11min 36s, sys: 1min 30s, total: 13min 7s
Wall time: 16min 1s
